# ZorgDeID — Quickstart

This notebook walks through all major features of the `zorgdeid` library:

1. Installation & import
2. Detect PII in plain text (`analyze.text`)
3. Anonymize plain text — all three modes (`guard.text`)
4. Work with documents — `.txt`, `.pdf`, `.docx` (`analyze.doc` / `guard.doc`)
5. Custom patterns
6. Config options — entity filters, score threshold

## 1. Installation

```bash
pip install zorgdeid
```

Or from the repo root:

```bash
pip install -e .
```

In [1]:
import zorgdeid
from zorgdeid import analyze, guard, custom_pattern, ALL_NL_ENTITY_TYPES

print("zorgdeid version:", zorgdeid.__version__)

zorgdeid version: 1.0.0


## 2. Detect PII in plain text

`analyze.text()` returns a list of findings. Each finding contains:

| Key | Description |
|-----|-------------|
| `type` | Entity label (e.g. `BSN`, `EMAIL_ADDRESS`) |
| `start` | Start character offset |
| `end` | End character offset |
| `score` | Confidence score (0.0 – 1.0) |

In [2]:
sample_text = (
    "Patiënt: Jan de Vries, BSN 999999990. "
    "IBAN: NL91 ABNA 0417 1643 00. "
    "E-mail: jan.devries@umcg.nl. "
    "Telefoon: 06-12345678."
)

findings = analyze.text(sample_text)

for f in findings:
    print(f"{f['type']:<20} {sample_text[f['start']:f['end']]:<30} score={f['score']:.2f}")

PERSON               Jan de Vries                   score=0.90
BSN                  999999990                      score=0.95
IBAN_CODE            NL91 ABNA 0417 1643 00         score=0.90
EMAIL_ADDRESS        jan.devries@umcg.nl            score=0.95
PHONE_NUMBER         06-12345678                    score=0.60


### All supported entity types

In [3]:
print(f"{len(ALL_NL_ENTITY_TYPES)} supported entity types:")
print(", ".join(sorted(ALL_NL_ENTITY_TYPES)))

19 supported entity types:
BSN, CREDIT_CARD, CVV, DATE, EMAIL_ADDRESS, GPS_COORDINATES, IBAN_CODE, IMEI, IP_ADDRESS, LICENCE_PLATE, LOCATION, MAC_ADDRESS, PASSPORT, PERSON, PHONE_NUMBER, TIME, URL, ZIPCODE, ZORGPOLIS_NUMBER


## 3. Anonymize plain text — three modes

`guard.text()` returns a dict with two keys:
- `guarded_text` — the text with PII replaced
- `findings` — list of detected spans (same structure as `analyze.text`, plus `original_text`)

### Mode: `anonymize` (default) — realistic synthetic replacements

In [4]:
result = guard.text(sample_text, config={"mode": "anonymize"})

print("Original:")
print(sample_text)
print()
print("Guarded:")
print(result["guarded_text"])

Original:
Patiënt: Jan de Vries, BSN 999999990. IBAN: NL91 ABNA 0417 1643 00. E-mail: jan.devries@umcg.nl. Telefoon: 06-12345678.

Guarded:
Patiënt: Jan Bakker, BSN 111222333. IBAN: NL20 INGB 0001 2345 67. E-mail: j.bakker@voorbeeld.nl. Telefoon: +31 6 87654321.


### Mode: `tag` — bracket labels

In [5]:
result_tag = guard.text(sample_text, config={"mode": "tag"})
print(result_tag["guarded_text"])

Patiënt: [PERSON], BSN [BSN]. IBAN: [IBAN_CODE]. E-mail: [EMAIL_ADDRESS]. Telefoon: [PHONE_NUMBER].


### Mode: `i_tag` — indexed bracket labels

In [6]:
result_itag = guard.text(sample_text, config={"mode": "i_tag"})
print(result_itag["guarded_text"])

Patiënt: [PERSON_1], BSN [BSN_1]. IBAN: [IBAN_CODE_1]. E-mail: [EMAIL_ADDRESS_1]. Telefoon: [PHONE_NUMBER_1].


### Inspect findings from `guard.text()`

In [7]:
for f in result["findings"]:
    print(f"{f['type']:<20} original='{f['original_text']}'  score={f['score']:.2f}")

PERSON               original='Jan de Vries'  score=0.90
BSN                  original='999999990'  score=0.95
IBAN_CODE            original='NL91 ABNA 0417 1643 00'  score=0.90
EMAIL_ADDRESS        original='jan.devries@umcg.nl'  score=0.95
PHONE_NUMBER         original='06-12345678'  score=0.60


## 4. Work with documents

`analyze.doc()` and `guard.doc()` accept `.txt`, `.pdf`, and `.docx` file paths.
They extract text automatically and run the same pipelines as their `.text()` counterparts.

The sample files are in `examples/files/`.

In [8]:
import os

FILES_DIR = os.path.join(os.path.dirname(os.getcwd()), "examples", "files")
# If running from examples/ folder already:
if not os.path.exists(FILES_DIR):
    FILES_DIR = os.path.join(os.getcwd(), "files")

TXT_FILE  = os.path.join(FILES_DIR, "medisch_verslag.txt")
PDF_FILE  = os.path.join(FILES_DIR, "medisch_verslag.pdf")
DOCX_FILE = os.path.join(FILES_DIR, "medisch_verslag.docx")

print("Files found:")
for path in [TXT_FILE, PDF_FILE, DOCX_FILE]:
    print(f"  {'✓' if os.path.exists(path) else '✗'}  {path}")

Files found:
  ✓  c:\Users\hibra\Desktop\github\dataguard\examples\files\medisch_verslag.txt
  ✓  c:\Users\hibra\Desktop\github\dataguard\examples\files\medisch_verslag.pdf
  ✓  c:\Users\hibra\Desktop\github\dataguard\examples\files\medisch_verslag.docx


### Analyze a `.txt` document

In [9]:
txt_findings = analyze.doc(TXT_FILE)

print(f"Found {len(txt_findings)} PII entities in medisch_verslag.txt:")
for f in txt_findings:
    print(f"  {f['type']:<20} score={f['score']:.2f}")

Found 14 PII entities in medisch_verslag.txt:
  PERSON               score=0.90
  DATE                 score=0.65
  BSN                  score=0.65
  ZIPCODE              score=0.65
  LOCATION             score=0.90
  PHONE_NUMBER         score=0.70
  EMAIL_ADDRESS        score=0.95
  DATE                 score=0.85
  LOCATION             score=0.85
  LOCATION             score=0.85
  PERSON               score=0.85
  IBAN_CODE            score=0.90
  URL                  score=0.70
  IP_ADDRESS           score=0.80


### Guard a `.pdf` document

In [10]:
pdf_result = guard.doc(PDF_FILE, config={"mode": "tag"})

print("Guarded PDF text (first 600 chars):")
print(pdf_result["guarded_text"][:600])

Guarded PDF text (first 600 chars):
Medisch Verslag: Cardiologie Overdracht Patiëntgegevens: Naam: [PERSON] Geslacht: Man Geboortedatum: [DATE] BSN: [BSN] Adres: Dorpsstraat 45, [ZIPCODE], [LOCATION] Telefoonnummer: [PHONE_NUMBER] E-mail: [EMAIL_ADDRESS] Klinische Notitie: Patiënt is op [DATE] opgenomen op de afdeling Spoedeisende Hulp wegens
aanhoudende pijn op de borst. De patiënt verklaarde dat de klachten begonnen tijdens een fietstocht nabij het [LOCATION] in [LOCATION].
 Tijdens het lichamelijk onderzoek vertoonde de patiënt een verhoogde bloeddruk. Er is een
afspraak ingepland voor een vervolgonderzoek bij de specialist, 


### Guard a `.docx` document

In [11]:
docx_result = guard.doc(DOCX_FILE, config={"mode": "anonymize"})

print("Guarded DOCX text (first 600 chars):")
print(docx_result["guarded_text"][:600])

Guarded DOCX text (first 600 chars):
Medisch Verslag: Cardiologie Overdracht

Patiëntgegevens:

Naam: Maria Janssen
Geslacht: Man
Geboortedatum: 27 april 1990
BSN: 111222333 
Adres: Dorpsstraat 45, 2500 GH, Den Haag
Telefoonnummer: +31 6 87654321

E-mail: j.bakker@voorbeeld.nl

Klinische Notitie:
Patiënt is op 14 januari 1983 opgenomen op de afdeling Spoedeisende Hulp wegens aanhoudende pijn op de borst. De patiënt verklaarde dat de klachten begonnen tijdens een fietstocht nabij het Rotterdam in Utrecht.

Tijdens het lichamelijk onderzoek vertoonde de patiënt een verhoogde bloeddruk. Er is een afspraak ingepland voor een vervolgo


## 5. Custom patterns

Use `custom_pattern()` to add your own regex-based entity types. Pass them via `config["custom_patterns"]`.

| Param | Description |
|-------|-------------|
| `name` | Entity label for this pattern |
| `regex` | Python regex |
| `score` | Confidence score (default `0.85`) |
| `context` | Words near the match that boost confidence |
| `anonymize_list` | Custom fake-value pool for `anonymize` mode |

In [12]:
employee_pattern = custom_pattern(
    name="EMPLOYEE_ID",
    regex=r"EMP-\d{4}",
    score=0.95,
    context=["medewerker", "employee", "personeelsnummer"],
    anonymize_list=["EMP-0001", "EMP-0002", "EMP-0003"],
)

text_with_employee = "Medewerker EMP-1234 heeft toegang tot het systeem."

findings = analyze.text(
    text_with_employee,
    config={"custom_patterns": [employee_pattern]},
)

print("Detected:", findings)

Detected: [{'type': 'EMPLOYEE_ID', 'start': 11, 'end': 19, 'score': 1.0}]


In [13]:
guarded = guard.text(
    text_with_employee,
    config={"custom_patterns": [employee_pattern], "mode": "anonymize"},
)

print("Original:", text_with_employee)
print("Guarded: ", guarded["guarded_text"])

Original: Medewerker EMP-1234 heeft toegang tot het systeem.
Guarded:  Medewerker EMP-0001 heeft toegang tot het systeem.


## 6. Config options & scoring

Every finding carries a `score` between 0 and 1, determined by a four-tier system:

| Tier | Condition |
|------|-----------|
| `base` | Regex match only, no additional evidence |
| `with_context` | A relevant keyword appears **in the same sentence** |
| `validated` | Algorithmic checksum passes (elfproef / mod-97 / Luhn) |
| `high_confidence` | Validation **and** context keyword present |

Context scoring is **sentence-aware** — keywords from other sentences do not influence the score.
Contradicting keywords (e.g. `factuurnummer` near a phone number) actively reduce confidence.

### Filter entities — keep only specific types

In [14]:
findings_bsn_only = analyze.text(
    sample_text,
    config={"set_entities": {"keep": ["BSN", "IBAN_CODE"]}},
)

print("Only BSN and IBAN:")
for f in findings_bsn_only:
    print(f"  {f['type']:<20} '{sample_text[f['start']:f['end']]}'") 

Only BSN and IBAN:
  BSN                  '999999990'
  IBAN_CODE            'NL91 ABNA 0417 1643 00'


### Filter entities — ignore specific types

In [15]:
findings_no_person = analyze.text(
    sample_text,
    config={"set_entities": {"ignore": ["PERSON"]}},
)

print("All entities except PERSON:")
for f in findings_no_person:
    print(f"  {f['type']:<20} '{sample_text[f['start']:f['end']]}'") 

All entities except PERSON:
  BSN                  '999999990'
  IBAN_CODE            'NL91 ABNA 0417 1643 00'
  EMAIL_ADDRESS        'jan.devries@umcg.nl'
  PHONE_NUMBER         '06-12345678'


### Score threshold — raise the bar for detections

In [16]:
high_confidence = analyze.text(sample_text, config={"score_threshold": 0.8})
all_detections  = analyze.text(sample_text, config={"score_threshold": 0.0})

print(f"All detections (threshold=0.0): {len(all_detections)} findings")
print(f"High confidence (threshold=0.8): {len(high_confidence)} findings")

All detections (threshold=0.0): 5 findings
High confidence (threshold=0.8): 4 findings


### Full medical record — end-to-end example

In [17]:
with open(TXT_FILE, encoding="utf-8") as fh:
    medical_text = fh.read()

result = guard.text(medical_text, config={"mode": "tag"})

print("=== Original ===")
print(medical_text)
print()
print("=== Guarded (tag mode) ===")
print(result["guarded_text"])
print()
print(f"Total PII spans detected: {len(result['findings'])}")

=== Original ===
﻿Medisch Verslag: Cardiologie Overdracht


Patiëntgegevens:


Naam: Jan-Willem van den Berg
Geslacht: Man
Geboortedatum: 12-05-1978
BSN: 138403980 
Adres: Dorpsstraat 45, 3521 CA, Utrecht
Telefoonnummer: +31 6 12345678


E-mail: j.vandenberg78@gmail.com


Klinische Notitie:
Patiënt is op 10 maart 2026 opgenomen op de afdeling Spoedeisende Hulp wegens aanhoudende pijn op de borst. De patiënt verklaarde dat de klachten begonnen tijdens een fietstocht nabij het Vondelpark in Amsterdam.


Tijdens het lichamelijk onderzoek vertoonde de patiënt een verhoogde bloeddruk. Er is een afspraak ingepland voor een vervolgonderzoek bij de specialist, Dr. Anisa Bakker. Voor de facturatie en administratieve afhandeling is het dossier gekoppeld aan IBAN: NL91 ABNA 0417 1643 00.


Behandelplan:
Patiënt dient rust te houden. Voor vragen kan contact worden opgenomen met de assistente via https://www.careons-clinic.nl/contact of via het directe interne IP-adres van de poli: 192.168.1.104.

